In [ ]:
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, START, END

# =========================================================
# 1. THE CHILD GRAPH (The Isolated Research Agent Team)
# =========================================================

# This is the private scratchpad memory for the Research Agent.
# The Parent Graph cannot see this schema or these keys.
class ResearchPrivateState(TypedDict):
    query: str
    raw_urls: list
    extracted_facts: list
    final_summary: str

def google_search_node(state: ResearchPrivateState):
    print("   🔍 [Sub-Graph: Researcher] Searching the web for sources...")
    return {"raw_urls": ["url1.com", "url2.com"]}

def scraping_node(state: ResearchPrivateState):
    print("   📄 [Sub-Graph: Researcher] Extracting text and facts from discovered URLs...")
    return {"extracted_facts": ["Fact A: LangGraph supports sub-graphs.", "Fact B: Sub-graphs isolate memory."]}

def summarizer_node(state: ResearchPrivateState):
    print("   ✍️ [Sub-Graph: Researcher] Compiling private facts into clean summary...")
    summary = f"Research complete. Summary: {', '.join(state['extracted_facts'])}"
    return {"final_summary": summary}

# Assemble the child graph map
research_builder = StateGraph(ResearchPrivateState)
research_builder.add_node("search", google_search_node)
research_builder.add_node("scrape", scraping_node)
research_builder.add_node("summarize", summarizer_node)

research_builder.add_edge(START, "search")
research_builder.add_edge("search", "scrape")
research_builder.add_edge("scrape", "summarize")
research_builder.add_edge("summarize", END)

# Compile the child graph. It is now a fully sealed black box agent.
research_agent_subgraph = research_builder.compile()


# =========================================================
# 2. THE PARENT GRAPH (The Executive Orchestrator)
# =========================================================

# The main executive memory only tracks high-level corporate deliverables
class ExecutiveState(TypedDict):
    project_topic: str
    executive_brief: str

def writer_node(state: ExecutiveState):
    print("👔 [Parent Graph: Executive Writer] Transforming raw research into a corporate brief...")
    # It reads the research summary deposited by the sub-graph agent
    brief = f"OFFICIAL CORPORATE BRIEF CASE STUDY:\n--> {state['executive_brief']}"
    return {"executive_brief": brief}


# =========================================================
# 3. WIRING THEM TOGETHER WITH AN EXPLICIT MEMORY BRIDGE
# =========================================================
parent_builder = StateGraph(ExecutiveState)

# Here is the structural distinction:
# We register the entire compiled compiled research_agent_subgraph as a SINGLE node inside the parent!
parent_builder.add_node("research_agent_node", research_agent_subgraph)
parent_builder.add_node("writer_node", writer_node)

parent_builder.add_edge(START, "research_agent_node")
parent_builder.add_edge("research_agent_node", "writer_node")
parent_builder.add_edge("writer_node", END)

# Compile the master system
enterprise_orchestrator = parent_builder.compile()
print("🏢 Hierarchical Multi-Agent System successfully compiled!")

# =========================================================
# 4. EXECUTION
# =========================================================
print("\n🚀 Injecting high-level task into Executive Graph...")
initial_request = {
    "project_topic": "LangGraph Architecture Insights",
    "executive_brief": "" # Empty, waiting for the research agent node to fill it
}

# LangGraph automatically handles cross-compiling states when entering/leaving sub-graphs
# If keys match or have clear pipelines, values pass across boundaries.
final_state = enterprise_orchestrator.invoke(
    {"project_topic": "LangGraph Architecture Insights", "executive_brief": ""}
)

print("\n--- FINAL MASTER OUTPUT ---")
print(final_state["executive_brief"])